In [4]:
import random
from pathlib import Path
import numpy as np

def generate_random_qasm_circuit(
    num_qubits: int,
    depth: int,
    single_qubit_gate_percentage: float,
    two_qubit_gate_percentage: float,
    output_path: Path | None = None,
    seed: int | None = None
) -> str:
    """
    Generate a random QASM 2.0 circuit.
    
    Parameters:
    -----------
    num_qubits: Number of qubits in the circuit
    depth: Number of layers (even layers = single-qubit, odd layers = two-qubit)
    single_qubit_gate_percentage: Percentage of qubits to apply single-qubit gates to (0.0-1.0)
    two_qubit_gate_percentage: Percentage of qubit pairs to apply CNOT gates to (0.0-1.0)
    output_path: Optional path to save the QASM file
    seed: Random seed for reproducibility
    
    Returns:
    --------
    qasm_string: The generated QASM circuit as a string
    """
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)
    
    # Start building QASM content
    qasm_lines = [
        "//",
        "//",
        "//",
        "//",
        "//",
        "",
        "OPENQASM 2.0;",
        'include "qelib1.inc";',
        "",
        f"qreg q[{num_qubits}];",
        ""
    ]
    
    # Generate circuit layers
    for layer in range(depth):
        if layer % 2 == 0:
            # Even layers: single-qubit gates
            num_gates = int(num_qubits * single_qubit_gate_percentage)
            if num_gates > 0:
                selected_qubits = random.sample(range(num_qubits), num_gates)
                
                for qubit in sorted(selected_qubits):
                    # Randomly choose between u1, u2, u3 gates with random parameters
                    gate_type = random.choice(['u1', 'u2', 'u3'])
                    
                    if gate_type == 'u1':
                        theta = round(random.uniform(0, 2 * np.pi), 2)
                        qasm_lines.append(f"u1({theta}) q[{qubit}];")
                    elif gate_type == 'u2':
                        phi = round(random.uniform(0, 2 * np.pi), 2)
                        lam = round(random.uniform(0, 2 * np.pi), 2)
                        qasm_lines.append(f"u2({phi},{lam}) q[{qubit}];")
                    else:  # u3
                        theta = round(random.uniform(0, np.pi), 2)
                        phi = round(random.uniform(0, 2 * np.pi), 2)
                        lam = round(random.uniform(0, 2 * np.pi), 2)
                        qasm_lines.append(f"u3({theta},{phi},{lam}) q[{qubit}];")
                
                qasm_lines.append("")  # Empty line after layer
        
        else:
            # Odd layers: two-qubit gates (CNOT)
            # Generate random pairs
            available_qubits = list(range(num_qubits))
            pairs = []
            
            # Calculate how many pairs to generate
            max_pairs = num_qubits // 2
            num_pairs = int(max_pairs * two_qubit_gate_percentage)
            
            if num_pairs > 0:
                random.shuffle(available_qubits)
                
                for i in range(num_pairs):
                    if len(available_qubits) >= 2:
                        control = available_qubits.pop()
                        target = available_qubits.pop()
                        pairs.append((control, target))
                
                # Sort pairs for consistent ordering
                for control, target in sorted(pairs):
                    qasm_lines.append(f"cx q[{control}],q[{target}];")
                
                qasm_lines.append("")  # Empty line after layer
    
    # Join all lines
    qasm_content = "\n".join(qasm_lines)
    
    # Save to file if path provided
    if output_path:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        with open(output_path, 'w') as f:
            f.write(qasm_content)
        print(f"Circuit saved to: {output_path}")
    
    return qasm_content

In [23]:
# Example usage: Generate a random circuit
num_qubits_list = range(5,61)

single_qubit_gate_percentage = 1.0 # % of qubits get single-qubit gates
two_qubit_gate_percentage = 0.0    # % of possible pairs get CNOT gates

output_title = f"generated_{single_qubit_gate_percentage}_{two_qubit_gate_percentage}"

path = f"../inputs/qasm_files/{output_title}"

# Create directory if it doesn't exist
if not Path(path).exists():
    Path(path).mkdir(parents=True, exist_ok=True)

output_dir = Path(path)

for num_qubits in num_qubits_list:
    depth = 2*num_qubits
    output_file = output_dir / f"{output_title}_{num_qubits}.qasm"

    # Generate the circuit
    circuit = generate_random_qasm_circuit(
        num_qubits=num_qubits,
        depth=depth,
        single_qubit_gate_percentage=single_qubit_gate_percentage,
        two_qubit_gate_percentage=two_qubit_gate_percentage,
        output_path=output_file,
        seed=42  # For reproducibility
    )

    print(f"\nGenerated circuit with {num_qubits} qubits and {depth} layers")
    print(f"\nFirst few lines:")
    print("\n".join(circuit.split("\n")[:20]))

Circuit saved to: ../inputs/qasm_files/generated_1.0_0.0/generated_1.0_0.0_5.qasm

Generated circuit with 5 qubits and 10 layers

First few lines:
//
//
//
//
//

OPENQASM 2.0;
include "qelib1.inc";

qreg q[5];

u1(0.88) q[0];
u1(4.25) q[1];
u3(0.27,2.65,0.19) q[2];
u1(1.46) q[3];
u3(0.08,1.25,4.08) q[4];

u1(4.77) q[0];
u1(4.39) q[1];
u2(1.75,1.35) q[2];
Circuit saved to: ../inputs/qasm_files/generated_1.0_0.0/generated_1.0_0.0_6.qasm

Generated circuit with 6 qubits and 12 layers

First few lines:
//
//
//
//
//

OPENQASM 2.0;
include "qelib1.inc";

qreg q[6];

u1(0.88) q[0];
u1(4.25) q[1];
u3(0.27,2.65,0.19) q[2];
u1(1.46) q[3];
u3(0.08,1.25,4.08) q[4];
u3(1.32,2.82,1.75) q[5];

u1(6.01) q[0];
u2(0.64,2.39) q[1];
Circuit saved to: ../inputs/qasm_files/generated_1.0_0.0/generated_1.0_0.0_7.qasm

Generated circuit with 7 qubits and 14 layers

First few lines:
//
//
//
//
//

OPENQASM 2.0;
include "qelib1.inc";

qreg q[7];

u3(0.32,4.65,3.43) q[0];
u3(1.33,0.19,1.37) q[1];
u3(1.89,3.53

In [8]:
# Cell 3: Debug circuit by reading QASM and visualizing
from qiskit import QuantumCircuit
import matplotlib.pyplot as plt

# Read the generated QASM file
qasm_file = output_file  # or specify a path directly
print(f"Reading circuit from: {qasm_file}")

# Load circuit from QASM
qc = QuantumCircuit.from_qasm_file(str(qasm_file))

print(f"\nCircuit info:")
print(f"  Qubits: {qc.num_qubits}")
print(f"  Depth: {qc.depth()}")
print(f"  Gates: {qc.size()}")
print(f"  Operations: {qc.count_ops()}")

# Draw the circuit (will display automatically in Jupyter)
qc.draw(output='mpl', fold=-1, scale=0.8)

Reading circuit from: ../inputs/qasm_files/generated_1.0_1.0/generated_1.0_1.0_60.qasm

Circuit info:
  Qubits: 60
  Depth: 120
  Gates: 5400
  Operations: OrderedDict({'cx': 1800, 'u2': 1215, 'u1': 1198, 'u3': 1187})
